# Prophet Tuning — Resume After Kaggle 12-Hour Timeout

This notebook **does not repeat feature screening** and **does not repeat completed tuning folds**.

Recovered from the previous Kaggle log:

- 32 / 32 folds for `weather + lags + is_holiday`
- 1 fold (`aug_2025`) for `weather + lags + prophet_uk_holidays`
- **33 / 64 tuning folds already completed**
- **31 tuning fits remain**

The notebook checkpoints after **every completed fold**.

June 2026 remains locked. This notebook is only for finishing Prophet tuning and saving the best Prophet configuration.


In [1]:
import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.serialize import model_to_json

pd.set_option('display.max_columns', 120)

INPUT_PATH = '/kaggle/input/datasets/kusalnirukshan/master-train-data-uk-demand/master_training_data.csv'

OUTPUT_FOLDER = (
    Path('/kaggle/working/prophet_outputs')
    if Path('/kaggle/working').exists()
    else Path('artifacts/prophet_notebook')
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

INPUT_PATH, OUTPUT_FOLDER


('/kaggle/input/datasets/kusalnirukshan/master-train-data-uk-demand/master_training_data.csv',
 PosixPath('/kaggle/working/prophet_outputs'))

## 1. Load the same master dataset

In [2]:
df = pd.read_csv(INPUT_PATH, low_memory=False)

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

df = (
    df
    .dropna(subset=['timestamp', 'demand_mw'])
    .sort_values('timestamp')
    .drop_duplicates(subset=['timestamp'], keep='last')
    .reset_index(drop=True)
)

print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Range:', df['timestamp'].min(), 'to', df['timestamp'].max())
print('Duplicate timestamps:', df['timestamp'].duplicated().sum())

df.head()


Rows: 144600
Columns: 51
Range: 2010-01-01 00:00:00 to 2026-06-30 23:00:00
Duplicate timestamps: 0


,timestamp,demand_mw,date,hour,day_of_week,day_of_month,month,weekend,season,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,shortwave_radiation,econ_year,econ_month,econ_industrial_production_index_lag1m,econ_gdp_index_lag1m,econ_cpi_index_lag1m,econ_unemployment_rate_lag1m,econ_economic_data_complete,is_holiday,holiday_name,cal_year,cal_month_num,cal_month_name,cal_day,cal_day_of_week,cal_day_of_week_num,cal_week_of_year,cal_quarter,cal_is_weekend,cal_season,cal_holiday_names,cal_holiday_regions,cal_is_bank_holiday_england_wales,cal_is_bank_holiday_scotland,cal_is_bank_holiday,cal_event_count,cal_event_names,cal_is_covid_lockdown,cal_is_general_election,cal_is_major_football,cal_is_event_day,cal_is_non_working_day
0,2010-01-01 00:00:00,36566.5,2010-01-01,0,4,1,1,0,winter,-0.76,83.4,-3.30,-5.33,0.01,0.00,1001.73,43.6,13.52,112.8,0.0,2010,1,99.9,79.8,88.2,7.7,1,1,NaN,2010,1,January,1,Friday,4,53,1,0,Winter,NaN,NaN,0,0,1,0,NaN,0,0,0,0,1
1,2010-01-01 01:00:00,35852.5,2010-01-01,1,4,1,1,0,winter,-1.02,83.3,-3.52,-5.67,0.00,0.00,1001.66,52.9,14.02,109.9,0.0,2010,1,99.9,79.8,88.2,7.7,1,1,NaN,2010,1,January,1,Friday,4,53,1,0,Winter,NaN,NaN,0,0,1,0,NaN,0,0,0,0,1
2,2010-01-01 02:00:00,34189.5,2010-01-01,2,4,1,1,0,winter,-1.17,83.5,-3.67,-5.79,0.02,0.01,1001.67,55.6,13.67,139.9,0.0,2010,1,99.9,79.8,88.2,7.7,1,1,NaN,2010,1,January,1,Friday,4,53,1,0,Winter,NaN,NaN,0,0,1,0,NaN,0,0,0,0,1
3,2010-01-01 03:00:00,32453.0,2010-01-01,3,4,1,1,0,winter,-1.22,84.5,-3.71,-5.85,0.05,0.05,1001.65,50.9,13.90,135.7,0.0,2010,1,99.9,79.8,88.2,7.7,1,1,NaN,2010,1,January,1,Friday,4,53,1,0,Winter,NaN,NaN,0,0,1,0,NaN,0,0,0,0,1
4,2010-01-01 04:00:00,30450.5,2010-01-01,4,4,1,1,0,winter,-1.22,85.8,-3.87,-5.85,0.05,0.04,1001.58,48.2,13.77,131.6,0.0,2010,1,99.9,79.8,88.2,7.7,1,1,NaN,2010,1,January,1,Friday,4,53,1,0,Winter,NaN,NaN,0,0,1,0,NaN,0,0,0,0,1


In [3]:
candidate_features = [
    # Weather
    'temperature_2m',
    'relative_humidity_2m',
    'dew_point_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'surface_pressure',
    'cloud_cover',
    'wind_speed_10m',
    'wind_direction_10m',
    'shortwave_radiation',

    # Economy
    'econ_industrial_production_index_lag1m',
    'econ_gdp_index_lag1m',
    'econ_cpi_index_lag1m',
    'econ_unemployment_rate_lag1m',

    # Calendar / events
    'weekend',
    'is_holiday',
    'cal_is_bank_holiday',
    'cal_is_bank_holiday_england_wales',
    'cal_is_bank_holiday_scotland',
    'cal_event_count',
    'cal_is_covid_lockdown',
    'cal_is_general_election',
    'cal_is_major_football',
    'cal_is_event_day',
    'cal_is_non_working_day',
]

candidate_features = [c for c in candidate_features if c in df.columns]

# Forward-fill only. Do NOT bfill time-series regressors.
for col in candidate_features:
    df[col] = pd.to_numeric(df[col], errors='coerce').ffill()

candidate_features


['temperature_2m',
 'relative_humidity_2m',
 'dew_point_2m',
 'apparent_temperature',
 'precipitation',
 'rain',
 'surface_pressure',
 'cloud_cover',
 'wind_speed_10m',
 'wind_direction_10m',
 'shortwave_radiation',
 'econ_industrial_production_index_lag1m',
 'econ_gdp_index_lag1m',
 'econ_cpi_index_lag1m',
 'econ_unemployment_rate_lag1m',
 'weekend',
 'is_holiday',
 'cal_is_bank_holiday',
 'cal_is_bank_holiday_england_wales',
 'cal_is_bank_holiday_scotland',
 'cal_event_count',
 'cal_is_covid_lockdown',
 'cal_is_general_election',
 'cal_is_major_football',
 'cal_is_event_day',
 'cal_is_non_working_day']

In [4]:
FINAL_TEST_START = pd.Timestamp('2026-06-01 00:00:00')

# Verify that shift(24) and shift(168) genuinely correspond to 24/168 hours.
hour_diffs = df['timestamp'].diff().dropna()

assert (
    hour_diffs == pd.Timedelta(hours=1)
).all(), (
    'Dataset is not perfectly hourly. '
    'Check missing timestamps before using row-based demand lags.'
)

# Historical demand lags suitable for rolling next-24-hour prediction.
df['demand_lag_24'] = df['demand_mw'].shift(24)
df['demand_lag_168'] = df['demand_mw'].shift(168)

# Final holdout split.
dev_df = df[df['timestamp'] < FINAL_TEST_START].copy()
final_test_df = df[df['timestamp'] >= FINAL_TEST_START].copy()

print(
    'Development:',
    len(dev_df),
    dev_df['timestamp'].min(),
    'to',
    dev_df['timestamp'].max(),
)

print(
    'FINAL TEST:',
    len(final_test_df),
    final_test_df['timestamp'].min(),
    'to',
    final_test_df['timestamp'].max(),
)

assert dev_df['timestamp'].max() < FINAL_TEST_START
assert final_test_df['timestamp'].min() == FINAL_TEST_START
assert final_test_df['timestamp'].max() <= pd.Timestamp('2026-06-30 23:00:00')

print('\nJune 2026 is isolated and must remain untouched until final model selection.')


Development: 143880 2010-01-01 00:00:00 to 2026-05-31 23:00:00
FINAL TEST: 720 2026-06-01 00:00:00 to 2026-06-30 23:00:00

June 2026 is isolated and must remain untouched until final model selection.


## 2. Prophet helpers and CV setup

In [5]:
def make_prophet_frame(source_df, regressors):
    required_columns = ['timestamp', 'demand_mw', *regressors]

    out = source_df[required_columns].copy()

    out = out.rename(
        columns={
            'timestamp': 'ds',
            'demand_mw': 'y',
        }
    )

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=['y', *regressors])

    return out


def calculate_metrics(actual, predicted):
    actual = pd.Series(actual).reset_index(drop=True)
    predicted = pd.Series(predicted).reset_index(drop=True)

    error = actual - predicted

    mae = error.abs().mean()
    rmse = np.sqrt((error ** 2).mean())
    mape = (
        error.abs() / actual.abs().clip(lower=1)
    ).mean() * 100

    ss_res = (error ** 2).sum()
    ss_tot = ((actual - actual.mean()) ** 2).sum()

    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

    return {
        'mae': round(float(mae), 4),
        'rmse': round(float(rmse), 4),
        'mape': round(float(mape), 4),
        'r2': round(float(r2), 4),
    }


def build_prophet_model(
    regressors,
    params=None,
    use_prophet_holidays=False,
):
    if params is None:
        params = {}

    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=False,
        seasonality_mode=params.get('seasonality_mode', 'additive'),
        changepoint_prior_scale=params.get('changepoint_prior_scale', 0.10),
        seasonality_prior_scale=params.get('seasonality_prior_scale', 10.0),
        holidays_prior_scale=params.get('holidays_prior_scale', 10.0),
    )

    model.add_seasonality(
        name='daily',
        period=1,
        fourier_order=params.get('daily_fourier_order', 16),
    )

    model.add_seasonality(
        name='weekly',
        period=7,
        fourier_order=params.get('weekly_fourier_order', 10),
    )

    model.add_seasonality(
        name='yearly',
        period=365.25,
        fourier_order=params.get('yearly_fourier_order', 12),
    )

    # Explicit, not automatic.
    if use_prophet_holidays:
        model.add_country_holidays(country_name='UK')

    for regressor in regressors:
        model.add_regressor(regressor)

    return model


In [6]:
screening_folds = [
    ('aug_2025', '2025-08-01 00:00:00', '2025-08-31 23:00:00'),
    ('nov_2025', '2025-11-01 00:00:00', '2025-11-30 23:00:00'),
    ('feb_2026', '2026-02-01 00:00:00', '2026-02-28 23:00:00'),
    ('may_2026', '2026-05-01 00:00:00', '2026-05-31 23:00:00'),
]

for fold_name, valid_start, valid_end in screening_folds:
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    assert valid_end < FINAL_TEST_START, (
        f'{fold_name} overlaps the final test set.'
    )

    assert valid_start <= valid_end

print('All cross-validation folds safely end before June 2026.')


All cross-validation folds safely end before June 2026.


In [7]:
weather_features = [
    'apparent_temperature',
    'relative_humidity_2m',
    'precipitation',
    'surface_pressure',
    'cloud_cover',
    'wind_speed_10m',
    'wind_direction_10m',
    'shortwave_radiation',
]

calendar_features = [
    'weekend',
    'is_holiday',
    'cal_event_count',
    'cal_is_covid_lockdown',
    'cal_is_general_election',
    'cal_is_major_football',
    'cal_is_non_working_day',
]

economic_features = [
    'econ_industrial_production_index_lag1m',
    'econ_gdp_index_lag1m',
    'econ_cpi_index_lag1m',
    'econ_unemployment_rate_lag1m',
]

lag_features = [
    'demand_lag_24',
    'demand_lag_168',
]

# Keep only features that actually exist.
weather_features = [c for c in weather_features if c in df.columns]
calendar_features = [c for c in calendar_features if c in df.columns]
economic_features = [c for c in economic_features if c in df.columns]

base_params = {
    'daily_fourier_order': 16,
    'weekly_fourier_order': 10,
    'yearly_fourier_order': 12,
    'changepoint_prior_scale': 0.10,
    'seasonality_prior_scale': 10.0,
    'seasonality_mode': 'additive',
}

print('Weather:', weather_features)
print('Calendar:', calendar_features)
print('Economy:', economic_features)
print('Lags:', lag_features)


Weather: ['apparent_temperature', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation']
Calendar: ['weekend', 'is_holiday', 'cal_event_count', 'cal_is_covid_lockdown', 'cal_is_general_election', 'cal_is_major_football', 'cal_is_non_working_day']
Economy: ['econ_industrial_production_index_lag1m', 'econ_gdp_index_lag1m', 'econ_cpi_index_lag1m', 'econ_unemployment_rate_lag1m']
Lags: ['demand_lag_24', 'demand_lag_168']


In [8]:
screening_variants = {
    # Pure Prophet temporal structure
    'seasonality_only': {
        'regressors': [],
        'use_prophet_holidays': False,
    },

    'prophet_uk_holidays': {
        'regressors': [],
        'use_prophet_holidays': True,
    },

    # Weather
    'weather_only': {
        'regressors': weather_features,
        'use_prophet_holidays': False,
    },

    'weather + prophet_uk_holidays': {
        'regressors': weather_features,
        'use_prophet_holidays': True,
    },

    # Historical demand
    'weather + lag24': {
        'regressors': weather_features + ['demand_lag_24'],
        'use_prophet_holidays': False,
    },

    'weather + lag168': {
        'regressors': weather_features + ['demand_lag_168'],
        'use_prophet_holidays': False,
    },

    'weather + lag24 + lag168': {
        'regressors': weather_features + [
            'demand_lag_24',
            'demand_lag_168',
        ],
        'use_prophet_holidays': False,
    },

    'weather + lags + is_holiday': {
        'regressors': weather_features + [
            'demand_lag_24',
            'demand_lag_168',
            'is_holiday',
        ],
        'use_prophet_holidays': False,
    },

    'weather + lags + prophet_uk_holidays': {
        'regressors': weather_features + [
            'demand_lag_24',
            'demand_lag_168',
        ],
        'use_prophet_holidays': True,
    },
}

# Add individual calendar candidates.
for feature in calendar_features:
    screening_variants[f'weather + {feature}'] = {
        'regressors': weather_features + [feature],
        'use_prophet_holidays': False,
    }

# Add individual economic candidates.
for feature in economic_features:
    screening_variants[f'weather + {feature}'] = {
        'regressors': weather_features + [feature],
        'use_prophet_holidays': False,
    }

print('Number of screening variants:', len(screening_variants))
list(screening_variants.keys())


Number of screening variants: 20


['seasonality_only',
 'prophet_uk_holidays',
 'weather_only',
 'weather + prophet_uk_holidays',
 'weather + lag24',
 'weather + lag168',
 'weather + lag24 + lag168',
 'weather + lags + is_holiday',
 'weather + lags + prophet_uk_holidays',
 'weather + weekend',
 'weather + is_holiday',
 'weather + cal_event_count',
 'weather + cal_is_covid_lockdown',
 'weather + cal_is_general_election',
 'weather + cal_is_major_football',
 'weather + cal_is_non_working_day',
 'weather + econ_industrial_production_index_lag1m',
 'weather + econ_gdp_index_lag1m',
 'weather + econ_cpi_index_lag1m',
 'weather + econ_unemployment_rate_lag1m']

In [9]:
def evaluate_one_fold(
    model_name,
    fold_name,
    regressors,
    params=None,
    use_prophet_holidays=False,
):
    if params is None:
        params = base_params

    fold_map = {
        name: (start, end)
        for name, start, end in screening_folds
    }

    valid_start, valid_end = fold_map[fold_name]
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    fold_train = dev_df[
        dev_df['timestamp'] < valid_start
    ].copy()

    fold_valid = dev_df[
        (dev_df['timestamp'] >= valid_start)
        & (dev_df['timestamp'] <= valid_end)
    ].copy()

    train = make_prophet_frame(fold_train, regressors)
    valid = make_prophet_frame(fold_valid, regressors)

    if train.empty or valid.empty:
        raise ValueError(
            f'{model_name} / {fold_name}: empty train or validation frame.'
        )

    model = build_prophet_model(
        regressors=regressors,
        params=params,
        use_prophet_holidays=use_prophet_holidays,
    )

    print(
        f'\n{model_name} | {fold_name} | '
        f'{len(regressors)} regressors | '
        f'train={len(train)} valid={len(valid)}'
    )

    model.fit(train[['ds', 'y', *regressors]])

    forecast = model.predict(
        valid[['ds', *regressors]]
    )

    predictions = (
        valid[['ds', 'y']]
        .merge(
            forecast[['ds', 'yhat']],
            on='ds',
            how='inner',
        )
    )

    metrics = calculate_metrics(
        predictions['y'],
        predictions['yhat'],
    )

    print(metrics)
    return metrics


## 3. Restore the 33 completed tuning folds from the previous run

In [10]:
recovered_tuning_rows = [{'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'aug_2025', 'mae': 1068.3702, 'rmse': 1408.4746, 'mape': 4.9858, 'r2': 0.802}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'nov_2025', 'mae': 1200.9059, 'rmse': 1541.7743, 'mape': 4.0727, 'r2': 0.9341}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'feb_2026', 'mae': 1295.3503, 'rmse': 1771.9448, 'mape': 4.2527, 'r2': 0.9018}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'may_2026', 'mae': 1051.8772, 'rmse': 1395.3331, 'mape': 4.8265, 'r2': 0.8211}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'aug_2025', 'mae': 1054.9259, 'rmse': 1389.0176, 'mape': 4.9326, 'r2': 0.8075}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'nov_2025', 'mae': 1192.9037, 'rmse': 1528.0188, 'mape': 4.0235, 'r2': 0.9353}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'feb_2026', 'mae': 1252.3714, 'rmse': 1722.5748, 'mape': 4.1182, 'r2': 0.9072}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'may_2026', 'mae': 1054.7188, 'rmse': 1395.9667, 'mape': 4.8444, 'r2': 0.821}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'aug_2025', 'mae': 1068.057, 'rmse': 1408.2977, 'mape': 4.9842, 'r2': 0.8021}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'nov_2025', 'mae': 1200.7376, 'rmse': 1541.6219, 'mape': 4.072, 'r2': 0.9341}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'feb_2026', 'mae': 1294.8144, 'rmse': 1771.0899, 'mape': 4.251, 'r2': 0.9019}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'may_2026', 'mae': 1052.3016, 'rmse': 1395.782, 'mape': 4.8285, 'r2': 0.821}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'aug_2025', 'mae': 1055.2185, 'rmse': 1389.3092, 'mape': 4.9337, 'r2': 0.8074}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'nov_2025', 'mae': 1192.7636, 'rmse': 1527.8504, 'mape': 4.0229, 'r2': 0.9353}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'feb_2026', 'mae': 1254.3739, 'rmse': 1725.7876, 'mape': 4.1254, 'r2': 0.9069}, {'config_id': 'weather + lags + is_holiday_cp0.05_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'may_2026', 'mae': 1055.3932, 'rmse': 1396.5636, 'mape': 4.8476, 'r2': 0.8208}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'aug_2025', 'mae': 1068.036, 'rmse': 1407.8359, 'mape': 4.9848, 'r2': 0.8022}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'nov_2025', 'mae': 1200.1117, 'rmse': 1541.0609, 'mape': 4.0698, 'r2': 0.9342}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'feb_2026', 'mae': 1295.2223, 'rmse': 1771.6967, 'mape': 4.2522, 'r2': 0.9018}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'may_2026', 'mae': 1052.529, 'rmse': 1396.0015, 'mape': 4.8295, 'r2': 0.821}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'aug_2025', 'mae': 1055.4753, 'rmse': 1389.424, 'mape': 4.9358, 'r2': 0.8074}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'nov_2025', 'mae': 1192.9393, 'rmse': 1528.0204, 'mape': 4.0233, 'r2': 0.9353}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'feb_2026', 'mae': 1255.0152, 'rmse': 1726.613, 'mape': 4.1273, 'r2': 0.9068}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp5.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'multiplicative', 'fold': 'may_2026', 'mae': 1058.0399, 'rmse': 1399.0364, 'mape': 4.8596, 'r2': 0.8202}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'aug_2025', 'mae': 1069.2031, 'rmse': 1406.5303, 'mape': 4.9942, 'r2': 0.8026}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'nov_2025', 'mae': 1199.0728, 'rmse': 1539.8646, 'mape': 4.0664, 'r2': 0.9343}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'feb_2026', 'mae': 1294.628, 'rmse': 1770.7562, 'mape': 4.2502, 'r2': 0.9019}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_additive', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'additive', 'fold': 'may_2026', 'mae': 1052.2447, 'rmse': 1395.7143, 'mape': 4.828, 'r2': 0.821}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'aug_2025', 'mae': 1055.269, 'rmse': 1389.3131, 'mape': 4.9346, 'r2': 0.8074}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'nov_2025', 'mae': 1192.9148, 'rmse': 1527.9771, 'mape': 4.0233, 'r2': 0.9353}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'feb_2026', 'mae': 1253.7709, 'rmse': 1724.8746, 'mape': 4.1232, 'r2': 0.907}, {'config_id': 'weather + lags + is_holiday_cp0.1_sp10.0_multiplicative', 'variant': 'weather + lags + is_holiday', 'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 10.0, 'seasonality_mode': 'multiplicative', 'fold': 'may_2026', 'mae': 1055.0289, 'rmse': 1396.2629, 'mape': 4.8458, 'r2': 0.8209}, {'config_id': 'weather + lags + prophet_uk_holidays_cp0.05_sp5.0_additive', 'variant': 'weather + lags + prophet_uk_holidays', 'changepoint_prior_scale': 0.05, 'seasonality_prior_scale': 5.0, 'seasonality_mode': 'additive', 'fold': 'aug_2025', 'mae': 1067.5552, 'rmse': 1506.1696, 'mape': 5.0086, 'r2': 0.7736}]

tuning_fold_df = pd.DataFrame(recovered_tuning_rows)
print('Recovered completed folds:', len(tuning_fold_df))
display(tuning_fold_df.tail())


Recovered completed folds: 33


,config_id,variant,changepoint_prior_scale,seasonality_prior_scale,seasonality_mode,fold,mae,rmse,mape,r2
28,weather + lags + is_holiday_cp0.1_sp10.0_multi...,weather + lags + is_holiday,0.10,10.0,multiplicative,aug_2025,1055.2690,1389.3131,4.9346,0.8074
29,weather + lags + is_holiday_cp0.1_sp10.0_multi...,weather + lags + is_holiday,0.10,10.0,multiplicative,nov_2025,1192.9148,1527.9771,4.0233,0.9353
30,weather + lags + is_holiday_cp0.1_sp10.0_multi...,weather + lags + is_holiday,0.10,10.0,multiplicative,feb_2026,1253.7709,1724.8746,4.1232,0.9070
31,weather + lags + is_holiday_cp0.1_sp10.0_multi...,weather + lags + is_holiday,0.10,10.0,multiplicative,may_2026,1055.0289,1396.2629,4.8458,0.8209
32,weather + lags + prophet_uk_holidays_cp0.05_sp...,weather + lags + prophet_uk_holidays,0.05,5.0,additive,aug_2025,1067.5552,1506.1696,5.0086,0.7736


## 4. Resume only unfinished tuning folds

In [11]:
import time

top_variant_names = [
    'weather + lags + is_holiday',
    'weather + lags + prophet_uk_holidays',
]

tuning_grid = {
    'changepoint_prior_scale': [0.05, 0.10],
    'seasonality_prior_scale': [5.0, 10.0],
    'seasonality_mode': ['additive', 'multiplicative'],
}

CHECKPOINT_PATH = OUTPUT_FOLDER / 'prophet_tuning_folds_checkpoint.csv'

# If this resume notebook itself was already run partially, merge its checkpoint.
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    tuning_fold_df = pd.concat(
        [tuning_fold_df, checkpoint_df],
        ignore_index=True,
    )

# Keep one row per completed config/fold pair.
tuning_fold_df = (
    tuning_fold_df
    .drop_duplicates(
        subset=['config_id', 'fold'],
        keep='last',
    )
    .reset_index(drop=True)
)

completed_pairs = set(
    zip(
        tuning_fold_df['config_id'],
        tuning_fold_df['fold'],
    )
)

all_fold_names = [x[0] for x in screening_folds]

total_expected = (
    len(top_variant_names)
    * len(tuning_grid['changepoint_prior_scale'])
    * len(tuning_grid['seasonality_prior_scale'])
    * len(tuning_grid['seasonality_mode'])
    * len(all_fold_names)
)

print(f'Already completed: {len(completed_pairs)} / {total_expected}')
print(f'Remaining: {total_expected - len(completed_pairs)} fits')

# Safety margin below Kaggle's 12-hour hard limit.
RUN_TIME_BUDGET_HOURS = 10.5
run_started = time.time()
stopped_for_time_budget = False

for model_name in top_variant_names:
    config = screening_variants[model_name]

    for (
        changepoint_prior_scale,
        seasonality_prior_scale,
        seasonality_mode,
    ) in product(
        tuning_grid['changepoint_prior_scale'],
        tuning_grid['seasonality_prior_scale'],
        tuning_grid['seasonality_mode'],
    ):
        params = {
            **base_params,
            'changepoint_prior_scale': changepoint_prior_scale,
            'seasonality_prior_scale': seasonality_prior_scale,
            'seasonality_mode': seasonality_mode,
        }

        config_id = (
            f'{model_name}'
            f'_cp{changepoint_prior_scale}'
            f'_sp{seasonality_prior_scale}'
            f'_{seasonality_mode}'
        )

        for fold_name in all_fold_names:
            key = (config_id, fold_name)

            if key in completed_pairs:
                print('Skipping completed:', config_id, '|', fold_name)
                continue

            elapsed_hours = (time.time() - run_started) / 3600
            if elapsed_hours >= RUN_TIME_BUDGET_HOURS:
                print(
                    '\nStopping safely before Kaggle timeout. '
                    'Checkpoint has been preserved.'
                )
                stopped_for_time_budget = True
                break

            metrics = evaluate_one_fold(
                model_name=config_id,
                fold_name=fold_name,
                regressors=config['regressors'],
                params=params,
                use_prophet_holidays=config['use_prophet_holidays'],
            )

            new_row = {
                'config_id': config_id,
                'variant': model_name,
                'changepoint_prior_scale': changepoint_prior_scale,
                'seasonality_prior_scale': seasonality_prior_scale,
                'seasonality_mode': seasonality_mode,
                'fold': fold_name,
                **metrics,
            }

            tuning_fold_df = pd.concat(
                [
                    tuning_fold_df,
                    pd.DataFrame([new_row]),
                ],
                ignore_index=True,
            )

            completed_pairs.add(key)

            # Critical: checkpoint after EVERY fold.
            tuning_fold_df.to_csv(
                CHECKPOINT_PATH,
                index=False,
            )

            print(
                f'Checkpoint saved — '
                f'{len(completed_pairs)} / {total_expected} folds complete'
            )

        if stopped_for_time_budget:
            break

    if stopped_for_time_budget:
        break

print('\nResume loop finished.')
print(f'Completed: {len(completed_pairs)} / {total_expected}')
print('Checkpoint:', CHECKPOINT_PATH)


Already completed: 33 / 64
Remaining: 31 fits
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_additive | aug_2025
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_additive | nov_2025
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_additive | feb_2026
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_additive | may_2026
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_multiplicative | aug_2025
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_multiplicative | nov_2025
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_multiplicative | feb_2026
Skipping completed: weather + lags + is_holiday_cp0.05_sp5.0_multiplicative | may_2026
Skipping completed: weather + lags + is_holiday_cp0.05_sp10.0_additive | aug_2025
Skipping completed: weather + lags + is_holiday_cp0.05_sp10.0_additive | nov_2025
Skipping completed: weather + lags + is_holiday_cp0.05_sp10.0_additive | feb_2026
Skipping completed: weather + lags +

16:52:21 - cmdstanpy - INFO - Chain [1] start processing
16:56:58 - cmdstanpy - INFO - Chain [1] done processing


{'mae': 1192.0905, 'rmse': 1530.917, 'mape': 4.0386, 'r2': 0.935}
Checkpoint saved — 34 / 64 folds complete

weather + lags + prophet_uk_holidays_cp0.05_sp5.0_additive | feb_2026 | 10 regressors | train=140832 valid=672


16:57:11 - cmdstanpy - INFO - Chain [1] start processing
17:01:22 - cmdstanpy - INFO - Chain [1] done processing


{'mae': 1286.4348, 'rmse': 1760.3974, 'mape': 4.2249, 'r2': 0.9031}
Checkpoint saved — 35 / 64 folds complete

weather + lags + prophet_uk_holidays_cp0.05_sp5.0_additive | may_2026 | 10 regressors | train=142968 valid=744


...

{'mae': 1185.0335, 'rmse': 1519.609, 'mape': 3.9904, 'r2': 0.936}
Checkpoint saved — 62 / 64 folds complete

weather + lags + prophet_uk_holidays_cp0.1_sp10.0_multiplicative | feb_2026 | 10 regressors | train=140832 valid=672


19:09:20 - cmdstanpy - INFO - Chain [1] start processing
19:13:33 - cmdstanpy - INFO - Chain [1] done processing


{'mae': 1246.787, 'rmse': 1716.0617, 'mape': 4.1008, 'r2': 0.9079}
Checkpoint saved — 63 / 64 folds complete

weather + lags + prophet_uk_holidays_cp0.1_sp10.0_multiplicative | may_2026 | 10 regressors | train=142968 valid=744


19:13:45 - cmdstanpy - INFO - Chain [1] start processing
19:18:12 - cmdstanpy - INFO - Chain [1] done processing


{'mae': 1051.4058, 'rmse': 1402.5352, 'mape': 4.8193, 'r2': 0.8193}
Checkpoint saved — 64 / 64 folds complete

Resume loop finished.
Completed: 64 / 64
Checkpoint: /kaggle/working/prophet_outputs/prophet_tuning_folds_checkpoint.csv


## 5. Build tuning summary from complete 4-fold configurations

In [12]:
fold_counts = (
    tuning_fold_df
    .groupby('config_id')['fold']
    .nunique()
)

complete_config_ids = fold_counts[
    fold_counts == len(screening_folds)
].index

complete_tuning_folds = tuning_fold_df[
    tuning_fold_df['config_id'].isin(complete_config_ids)
].copy()

tuning_summary = (
    complete_tuning_folds
    .groupby(
        [
            'config_id',
            'variant',
            'changepoint_prior_scale',
            'seasonality_prior_scale',
            'seasonality_mode',
        ],
        as_index=False,
    )
    .agg(
        mean_mae=('mae', 'mean'),
        mean_rmse=('rmse', 'mean'),
        mean_mape=('mape', 'mean'),
        mean_r2=('r2', 'mean'),
        std_rmse=('rmse', 'std'),
        folds_completed=('fold', 'nunique'),
    )
    .sort_values('mean_rmse')
    .reset_index(drop=True)
)

tuning_fold_df.to_csv(
    OUTPUT_FOLDER / 'prophet_tuning_folds.csv',
    index=False,
)

tuning_summary.to_csv(
    OUTPUT_FOLDER / 'prophet_tuning_summary.csv',
    index=False,
)

print(
    f'Complete configurations: '
    f'{len(tuning_summary)} / 16'
)

display(tuning_summary)


Complete configurations: 16 / 16


,config_id,variant,changepoint_prior_scale,seasonality_prior_scale,seasonality_mode,mean_mae,mean_rmse,mean_mape,mean_r2,std_rmse,folds_completed
0,weather + lags + is_holiday_cp0.05_sp5.0_multi...,weather + lags + is_holiday,0.05,5.0,multiplicative,1138.729950,1508.894475,4.479675,0.867750,156.149681,4
1,weather + lags + is_holiday_cp0.1_sp10.0_multi...,weather + lags + is_holiday,0.10,10.0,multiplicative,1139.245900,1509.606925,4.481725,0.867650,157.051061,4
2,weather + lags + is_holiday_cp0.05_sp10.0_mult...,weather + lags + is_holiday,0.05,10.0,multiplicative,1139.437300,1509.877700,4.482400,0.867600,157.392240,4
3,weather + lags + is_holiday_cp0.1_sp5.0_multip...,weather + lags + is_holiday,0.10,5.0,multiplicative,1140.367425,1510.773450,4.486500,0.867425,157.157062,4
4,weather + lags + is_holiday_cp0.1_sp10.0_additive,weather + lags + is_holiday,0.10,10.0,additive,1153.787150,1528.216350,4.534700,0.864950,174.475928,4
5,weather + lags + is_holiday_cp0.1_sp5.0_additive,weather + lags + is_holiday,0.10,5.0,additive,1153.974750,1529.148750,4.534075,0.864800,174.562697,4
6,weather + lags + is_holiday_cp0.05_sp10.0_addi...,weather + lags + is_holiday,0.05,10.0,additive,1153.977650,1529.197875,4.533925,0.864775,174.243848,4
7,weather + lags + is_holiday_cp0.05_sp5.0_additive,weather + lags + is_holiday,0.05,5.0,additive,1154.125900,1529.381700,4.534425,0.864750,174.716902,4
8,weather + lags + prophet_uk_holidays_cp0.05_sp...,weather + lags + prophet_uk_holidays,0.05,5.0,multiplicative,1135.112725,1532.252650,4.469275,0.860425,131.643777,4
9,weather + lags + prophet_uk_holidays_cp0.1_sp1...,weather + lags + prophet_uk_holidays,0.10,10.0,multiplicative,1135.043925,1532.264775,4.469950,0.860350,132.270127,4


## 6. Save best Prophet configuration when tuning is complete

In [13]:
EXPECTED_TUNING_FOLDS = 64

if len(completed_pairs) == EXPECTED_TUNING_FOLDS:
    best_row = tuning_summary.iloc[0]

    best_variant_name = best_row['variant']
    best_variant = screening_variants[best_variant_name]

    best_params = {
        **base_params,
        'changepoint_prior_scale': float(
            best_row['changepoint_prior_scale']
        ),
        'seasonality_prior_scale': float(
            best_row['seasonality_prior_scale']
        ),
        'seasonality_mode': best_row['seasonality_mode'],
    }

    best_prophet_config = {
        'model_type': 'Prophet',
        'variant': best_variant_name,
        'regressors': best_variant['regressors'],
        'use_prophet_holidays': bool(
            best_variant['use_prophet_holidays']
        ),
        'params': best_params,
        'selection_metric': 'mean_cv_rmse',
        'mean_cv_rmse': float(best_row['mean_rmse']),
        'mean_cv_mae': float(best_row['mean_mae']),
        'mean_cv_mape': float(best_row['mean_mape']),
        'mean_cv_r2': float(best_row['mean_r2']),
        'final_test_start': str(FINAL_TEST_START),
    }

    with open(
        OUTPUT_FOLDER / 'best_prophet_config.json',
        'w',
        encoding='utf-8',
    ) as f:
        json.dump(
            best_prophet_config,
            f,
            indent=2,
        )

    print('TUNING COMPLETE.')
    print('Best configuration:')
    display(best_prophet_config)

else:
    print(
        'Tuning is not fully complete yet. '
        'Do not select the final configuration until all 64 folds are present.'
    )
    print(
        f'Current progress: '
        f'{len(completed_pairs)} / {EXPECTED_TUNING_FOLDS}'
    )


TUNING COMPLETE.
Best configuration:


{'model_type': 'Prophet',
 'variant': 'weather + lags + is_holiday',
 'regressors': ['apparent_temperature',
  'relative_humidity_2m',
  'precipitation',
  'surface_pressure',
  'cloud_cover',
  'wind_speed_10m',
  'wind_direction_10m',
  'shortwave_radiation',
  'demand_lag_24',
  'demand_lag_168',
  'is_holiday'],
 'use_prophet_holidays': False,
 'params': {'daily_fourier_order': 16,
  'weekly_fourier_order': 10,
  'yearly_fourier_order': 12,
  'changepoint_prior_scale': 0.05,
  'seasonality_prior_scale': 5.0,
  'seasonality_mode': 'multiplicative'},
 'selection_metric': 'mean_cv_rmse',
 'mean_cv_rmse': 1508.894475,
 'mean_cv_mae': 1138.72995,
 'mean_cv_mape': 4.479675,
 'mean_cv_r2': 0.86775,
 'final_test_start': '2026-06-01 00:00:00'}

---

## Stop here before June

This resume notebook intentionally does **not** run the June 2026 holdout or production retraining.

Once Prophet tuning is complete, compare the best Prophet CV result with the other candidate models using the same development folds. Only after overall model selection should June be evaluated once.
